# 🛡️ IoT IDS — Two-Stage Attack Detection Pipeline
**Team:** Jinhong Lin · Elton Chang · Shiwei Jiang

## Architecture
```
Network Flow
     │
     ▼
┌─────────────────────────────────┐
│  STAGE 1 — Binary Classifier    │  Attack (0)  vs  Normal (1)
│  LogReg · RF · HGB · MLP · CNN  │
└──────────────┬──────────────────┘
               │  if ATTACK ↓
               ▼
┌─────────────────────────────────┐
│  STAGE 2 — Attack Type ID       │  udp · tcp · syn · ack
│  HGB  +  1D CNN (PyTorch)       │  combo · junk · scan · udpplain
└─────────────────────────────────┘
```

All outputs saved to:
`C:\Users\henda\OneDrive\Desktop\CNN_Group\Outputs\`


## Section 0 — Install & Imports

In [ ]:
# Run this cell first — installs any missing packages
import subprocess, sys
pkgs = ["torch", "scikit-learn", "pandas", "numpy",
        "matplotlib", "seaborn", "tqdm"]
for pkg in pkgs:
    subprocess.run([sys.executable, "-m", "pip", "install", pkg, "-q"],
                   capture_output=True)
print("✅ All packages ready")


In [ ]:
import os, time, warnings, json, re
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")          # saves figures to disk (no pop-up needed)
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from pathlib import Path
from tqdm import tqdm

# PyTorch
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

# Scikit-learn
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import (
    classification_report, confusion_matrix,
    f1_score, precision_score, recall_score,
    roc_auc_score, ConfusionMatrixDisplay,
)
from sklearn.inspection import permutation_importance

warnings.filterwarnings("ignore")
torch.manual_seed(42)
np.random.seed(42)
plt.rcParams.update({"figure.dpi": 110,
                     "axes.spines.top": False,
                     "axes.spines.right": False})

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"PyTorch  : {torch.__version__}")
print(f"Device   : {DEVICE}")


In [ ]:
\
# ── Paths ─────────────────────────────────────────────────────────────────
DATA_DIR = Path(r"C:\Users\henda\OneDrive\Desktop\CNN_Group")
OUT      = Path(r"C:\Users\henda\OneDrive\Desktop\CNN_Group\Outputs")
OUT.mkdir(parents=True, exist_ok=True)

CSV1 = DATA_DIR / "BotNeTIoT-L01_label_NoDuplicates.csv"
CSV2 = DATA_DIR / "BoTNeTIoT-L01-v2.csv"

# ── Config ─────────────────────────────────────────────────────────────────
SAMPLE_N     = 400_000
SAMPLE_N2    = 500_000
RANDOM_STATE = 42
TEST_SIZE    = 0.15
VAL_FRAC     = 0.15 / (1 - 0.15)

print(f"Data dir : {DATA_DIR}")
print(f"Output   : {OUT}")
for p, label in [(CSV1,"Dataset 1"), (CSV2,"Dataset 2")]:
    if p.exists():
        print(f"  ✅ {label}: {p.name}  ({p.stat().st_size/1e6:.0f} MB)")
    else:
        print(f"  ❌ {label} NOT FOUND: {p.name}")
        print(f"     → Place the CSV in {DATA_DIR}")


## Section 1 — Data Loading

In [ ]:
def stratified_sample_csv(path, n_samples, label_col="label",
                           chunksize=150_000, random_state=42):
    path = Path(path)
    print(f"  📂 {path.name}  ({path.stat().st_size/1e6:.0f} MB)")
    if n_samples is None:
        df = pd.read_csv(path)
        print(f"  ✅ {len(df):,} rows (full)")
        return df

    print("  Scanning labels …")
    all_labels = pd.concat(
        [chunk[label_col] for chunk in
         pd.read_csv(path, usecols=[label_col], chunksize=chunksize)],
        ignore_index=True).values

    classes, counts = np.unique(all_labels, return_counts=True)
    fracs = counts / counts.sum()
    rng   = np.random.RandomState(random_state)
    chosen = set()
    for c, f in zip(classes, fracs):
        idx_c = np.where(all_labels == c)[0]
        n_c   = min(int(n_samples * f), len(idx_c))
        chosen.update(rng.choice(idx_c, size=n_c, replace=False).tolist())

    print(f"  Reading {len(chosen):,} selected rows …")
    chunks_out, row_ptr = [], 0
    for chunk in pd.read_csv(path, chunksize=chunksize):
        mask = np.array([(row_ptr + i) in chosen for i in range(len(chunk))])
        if mask.any():
            chunks_out.append(chunk[mask])
        row_ptr += len(chunk)

    df = pd.concat(chunks_out, ignore_index=True)
    dist = {int(c): int(n) for c, n in
            zip(*np.unique(df[label_col], return_counts=True))}
    print(f"  ✅ {len(df):,} rows  |  classes: {dist}")
    return df


src1 = CSV1 if CSV1.exists() else CSV2
print(f"Loading Dataset 1 from {src1.name} …")
df1 = stratified_sample_csv(src1, SAMPLE_N)

print("\nLoading Dataset 2 …")
df2 = stratified_sample_csv(CSV2, SAMPLE_N2)

for _df in [df1, df2]:
    if "Unnamed: 0" in _df.columns:
        _df.drop(columns=["Unnamed: 0"], inplace=True)

print(f"\n✅ df1: {df1.shape}  |  df2: {df2.shape}")


## Section 2 — Preprocessing

In [ ]:
FEATURE_COLS = [
    "MI_dir_L0.1_weight","MI_dir_L0.1_mean","MI_dir_L0.1_variance",
    "H_L0.1_weight","H_L0.1_mean","H_L0.1_variance",
    "HH_L0.1_weight","HH_L0.1_mean","HH_L0.1_std",
    "HH_L0.1_magnitude","HH_L0.1_radius","HH_L0.1_covariance","HH_L0.1_pcc",
    "HH_jit_L0.1_weight","HH_jit_L0.1_mean","HH_jit_L0.1_variance",
    "HpHp_L0.1_weight","HpHp_L0.1_mean","HpHp_L0.1_std",
    "HpHp_L0.1_magnitude","HpHp_L0.1_radius","HpHp_L0.1_covariance","HpHp_L0.1_pcc",
]
# Drop 4 perfectly redundant features (confirmed r=1.00)
DROP_REDUNDANT = ["H_L0.1_weight","H_L0.1_mean","H_L0.1_variance","HH_jit_L0.1_weight"]
USE_FEATS = [f for f in FEATURE_COLS if f not in DROP_REDUNDANT]
print(f"Features: {len(FEATURE_COLS)} → {len(USE_FEATS)} after dropping {len(DROP_REDUNDANT)} redundant")

def log1p_tf(X_df, cols):
    X = X_df.copy()
    for c in cols:
        if c in X.columns:
            X[c] = np.log1p(np.abs(X[c]))
    return X

SKEW_FEATS = [f for f in USE_FEATS if abs(df1[f].skew()) > 1]
print(f"log1p applied to {len(SKEW_FEATS)} skewed features")

# ── Stage 1 splits (binary, from df1) ─────────────────────────────────────
X_all = df1[USE_FEATS].copy()
y_all = df1["label"].copy()

X_tv, X_test, y_tv, y_test = train_test_split(
    X_all, y_all, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y_all)
X_train, X_val, y_train, y_val = train_test_split(
    X_tv, y_tv, test_size=VAL_FRAC, random_state=RANDOM_STATE, stratify=y_tv)

scaler1 = StandardScaler()
X_train_s = scaler1.fit_transform(log1p_tf(X_train, SKEW_FEATS))
X_val_s   = scaler1.transform(log1p_tf(X_val,   SKEW_FEATS))
X_test_s  = scaler1.transform(log1p_tf(X_test,  SKEW_FEATS))

print(f"Stage-1 → Train: {len(X_train):,} | Val: {len(X_val):,} | Test: {len(X_test):,}")

# ── Stage 2 splits (multiclass, from df2) ─────────────────────────────────
use2    = [f for f in USE_FEATS if f in df2.columns]
y2_sub  = df2["Attack_subType"] if "Attack_subType" in df2.columns else None
y2_fam  = df2["Attack"]         if "Attack"         in df2.columns else None

if y2_sub is not None:
    X2_all = log1p_tf(df2[use2], SKEW_FEATS)
    X2tv, X2te, y2tv, y2te = train_test_split(
        X2_all, y2_sub, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y2_sub)
    X2tr, X2val, y2tr, y2val = train_test_split(
        X2tv, y2tv, test_size=VAL_FRAC, random_state=RANDOM_STATE, stratify=y2tv)

    scaler2 = StandardScaler()
    X2tr_s  = scaler2.fit_transform(X2tr)
    X2val_s = scaler2.transform(X2val)
    X2te_s  = scaler2.transform(X2te)

    le = LabelEncoder()
    y2tr_enc  = le.fit_transform(y2tr)
    y2val_enc = le.transform(y2val)
    y2te_enc  = le.transform(y2te)
    N_CLASSES2 = len(le.classes_)
    print(f"Stage-2 → Train: {len(X2tr):,} | Test: {len(X2te):,}")
    print(f"Stage-2 classes ({N_CLASSES2}): {list(le.classes_)}")

print("\n✅ Preprocessing complete")


## Section 3 — 1D CNN Architecture (PyTorch)

Each sample's 19 flow statistics are treated as a **1D sequence**.
Conv1D filters slide across neighbouring features to learn co-occurring
patterns (e.g. high packet weight + high jitter variance = flood attack).

```
Input (batch, 19) → unsqueeze → (batch, 1, 19)
  Conv1d(1→64,  k=3, pad=1) → BN → ReLU
  Conv1d(64→128,k=3, pad=1) → BN → ReLU
  Conv1d(128→64,k=3, pad=1) → BN → ReLU
  AdaptiveAvgPool1d(1) → flatten
  Linear(64→64) → ReLU → Dropout(0.3)
  Linear(64→n_out)  [sigmoid binary / softmax multiclass]
```


In [ ]:
class CNN1D(nn.Module):
    """1D CNN for tabular IoT flow data."""
    def __init__(self, n_features: int, n_out: int = 1):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv1d(1,   64,  kernel_size=3, padding=1), nn.BatchNorm1d(64),  nn.ReLU(),
            nn.Conv1d(64,  128, kernel_size=3, padding=1), nn.BatchNorm1d(128), nn.ReLU(),
            nn.Conv1d(128, 64,  kernel_size=3, padding=1), nn.BatchNorm1d(64),  nn.ReLU(),
        )
        self.pool = nn.AdaptiveAvgPool1d(1)
        self.head = nn.Sequential(
            nn.Linear(64, 64), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(64, n_out),
        )
        self.n_out = n_out

    def forward(self, x):               # x: (B, F)
        x = x.unsqueeze(1)              # (B, 1, F)
        x = self.conv(x)                # (B, 64, F)
        x = self.pool(x).squeeze(-1)    # (B, 64)
        return self.head(x)             # (B, n_out)


def make_loader(X, y, batch_size=1024, shuffle=True):
    Xt = torch.tensor(X, dtype=torch.float32)
    yt = torch.tensor(y, dtype=torch.float32 if y.ndim == 1 and y.max() <= 1 else torch.long)
    return DataLoader(TensorDataset(Xt, yt), batch_size=batch_size, shuffle=shuffle)


def train_cnn(model, train_loader, val_loader, epochs=30, lr=1e-3,
              pos_weight=None, patience=5, task="binary"):
    """Train loop — works for binary (BCE) and multiclass (CE)."""
    model.to(DEVICE)
    opt   = optim.Adam(model.parameters(), lr=lr)
    sched = optim.lr_scheduler.ReduceLROnPlateau(opt, factor=0.5, patience=2, min_lr=1e-5)

    if task == "binary":
        criterion = nn.BCEWithLogitsLoss(
            pos_weight=torch.tensor([pos_weight], device=DEVICE) if pos_weight else None)
    else:
        criterion = nn.CrossEntropyLoss()

    history = {"train_loss":[], "val_loss":[], "val_metric":[]}
    best_val, best_state, no_improve = float("inf"), None, 0

    for epoch in range(epochs):
        # ── train ──
        model.train()
        t_loss = 0
        for Xb, yb in train_loader:
            Xb, yb = Xb.to(DEVICE), yb.to(DEVICE)
            opt.zero_grad()
            out = model(Xb)
            loss = criterion(out.squeeze(-1) if task=="binary" else out, yb)
            loss.backward(); opt.step()
            t_loss += loss.item() * len(Xb)
        t_loss /= len(train_loader.dataset)

        # ── val ──
        model.eval()
        v_loss, all_pred, all_true = 0, [], []
        with torch.no_grad():
            for Xb, yb in val_loader:
                Xb, yb = Xb.to(DEVICE), yb.to(DEVICE)
                out = model(Xb)
                loss = criterion(out.squeeze(-1) if task=="binary" else out, yb)
                v_loss += loss.item() * len(Xb)
                if task == "binary":
                    preds = (torch.sigmoid(out.squeeze(-1)) >= 0.5).long()
                else:
                    preds = out.argmax(dim=1)
                all_pred.extend(preds.cpu().numpy())
                all_true.extend(yb.cpu().numpy())
        v_loss /= len(val_loader.dataset)
        metric  = f1_score(all_true, all_pred, average="macro", zero_division=0)
        sched.step(v_loss)
        history["train_loss"].append(t_loss)
        history["val_loss"].append(v_loss)
        history["val_metric"].append(metric)

        if (epoch+1) % 5 == 0 or epoch == 0:
            print(f"  Epoch {epoch+1:3d}/{epochs}  "
                  f"train_loss={t_loss:.4f}  val_loss={v_loss:.4f}  "
                  f"val_F1={metric:.4f}")

        if v_loss < best_val:
            best_val, best_state, no_improve = v_loss, model.state_dict().copy(), 0
        else:
            no_improve += 1
            if no_improve >= patience:
                print(f"  Early stopping at epoch {epoch+1}")
                break

    model.load_state_dict(best_state)
    return history


def predict_cnn(model, X, task="binary", batch_size=2048):
    model.eval()
    Xt     = torch.tensor(X, dtype=torch.float32)
    loader = DataLoader(TensorDataset(Xt), batch_size=batch_size, shuffle=False)
    preds, probas = [], []
    with torch.no_grad():
        for (Xb,) in loader:
            Xb = Xb.to(DEVICE)
            out = model(Xb)
            if task == "binary":
                p = torch.sigmoid(out.squeeze(-1)).cpu().numpy()
                probas.extend(p); preds.extend((p >= 0.5).astype(int))
            else:
                p = torch.softmax(out, dim=1).cpu().numpy()
                probas.extend(p); preds.extend(p.argmax(axis=1))
    return np.array(preds), np.array(probas)


print("✅ CNN1D and training helpers defined")
# Quick architecture printout
_tmp = CNN1D(len(USE_FEATS), 1)
total_params = sum(p.numel() for p in _tmp.parameters() if p.requires_grad)
print(f"   Stage-1 CNN params: {total_params:,}")
del _tmp


## Section 4 — Stage 1: Binary Classification (Attack vs Normal)

In [ ]:
cw  = compute_class_weight("balanced", classes=np.array([0,1]), y=y_train)
cwd = {0: float(cw[0]), 1: float(cw[1])}
print(f"Class weights → Attack(0): {cw[0]:.3f}  |  Normal(1): {cw[1]:.3f}")

sklearn_models = {
    "LogReg": LogisticRegression(max_iter=500, random_state=RANDOM_STATE,
               class_weight="balanced", solver="lbfgs", C=1.0),
    "RF":     RandomForestClassifier(n_estimators=100, max_depth=20,
               random_state=RANDOM_STATE, class_weight="balanced", n_jobs=-1),
    "HGB":    HistGradientBoostingClassifier(max_iter=150, max_depth=8,
               learning_rate=0.05, random_state=RANDOM_STATE, class_weight="balanced"),
    "MLP":    MLPClassifier(hidden_layer_sizes=(256,128,64), activation="relu",
               solver="adam", alpha=1e-4, batch_size=1024, learning_rate_init=1e-3,
               max_iter=30, early_stopping=True, validation_fraction=0.1,
               n_iter_no_change=5, random_state=RANDOM_STATE),
}

s1_val = {}
for name, model in sklearn_models.items():
    print(f"\n── {name} {'─'*(38-len(name))}")
    t0 = time.time()
    model.fit(X_train_s, y_train)
    tt = time.time() - t0

    y_pred  = model.predict(X_val_s)
    y_proba = (model.predict_proba(X_val_s)[:,1]
               if hasattr(model,"predict_proba")
               else model.decision_function(X_val_s))
    t1 = time.time(); _ = model.predict(X_val_s)
    lat = (time.time()-t1)/len(X_val_s)*1e6

    s1_val[name] = dict(model=model,
        f1_attack=f1_score(y_val,y_pred,pos_label=0),
        f1_normal=f1_score(y_val,y_pred,pos_label=1),
        f1_macro =f1_score(y_val,y_pred,average="macro"),
        auc_roc  =roc_auc_score(y_val,y_proba),
        latency_us=lat, train_time=tt)
    print(f"  F1-Macro: {s1_val[name]['f1_macro']:.4f}  "
          f"AUC: {s1_val[name]['auc_roc']:.4f}  "
          f"Latency: {lat:.2f}μs  Train: {tt:.1f}s")

print("\n✅ Sklearn Stage-1 models done")


In [ ]:
# ── Stage 1 CNN (PyTorch) ─────────────────────────────────────────────────
pos_w = cw[1] / cw[0]          # upweight normal (minority)

cnn_s1    = CNN1D(X_train_s.shape[1], n_out=1)
tr_loader = make_loader(X_train_s, y_train.values.astype(np.float32))
vl_loader = make_loader(X_val_s,   y_val.values.astype(np.float32),   shuffle=False)

print("Training Stage-1 CNN (PyTorch) …")
t0 = time.time()
hist_s1 = train_cnn(cnn_s1, tr_loader, vl_loader,
                    epochs=30, lr=1e-3, pos_weight=pos_w,
                    patience=5, task="binary")
cnn_s1_time = time.time() - t0
print(f"\nDone in {cnn_s1_time:.1f}s  ({len(hist_s1['train_loss'])} epochs)")

cnn_val_pred, cnn_val_proba = predict_cnn(cnn_s1, X_val_s, task="binary")
cnn_lat_t = time.time()
_, _ = predict_cnn(cnn_s1, X_val_s[:500], task="binary")
cnn_lat = (time.time()-cnn_lat_t)/500*1e6

s1_val["CNN"] = dict(model=cnn_s1,
    f1_attack =f1_score(y_val, cnn_val_pred, pos_label=0),
    f1_normal =f1_score(y_val, cnn_val_pred, pos_label=1),
    f1_macro  =f1_score(y_val, cnn_val_pred, average="macro"),
    auc_roc   =roc_auc_score(y_val, cnn_val_proba),
    latency_us=cnn_lat, train_time=cnn_s1_time)
print(f"CNN Stage-1 → F1-Macro: {s1_val['CNN']['f1_macro']:.4f}  "
      f"AUC: {s1_val['CNN']['auc_roc']:.4f}  Latency: {cnn_lat:.2f}μs")


In [ ]:
# ── Stage-1 CNN Training Curves ──────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
epochs_ran = range(1, len(hist_s1["train_loss"])+1)
axes[0].plot(epochs_ran, hist_s1["train_loss"], label="train", color="#E74C3C")
axes[0].plot(epochs_ran, hist_s1["val_loss"],   label="val",   color="#3498DB")
axes[0].set_title("CNN Stage 1 — Loss",    fontweight="bold"); axes[0].legend()
axes[1].plot(epochs_ran, hist_s1["val_metric"], color="#2ECC71")
axes[1].set_title("CNN Stage 1 — Val F1-Macro", fontweight="bold")
plt.suptitle("Stage 1 CNN Training Curves", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig(OUT / "cnn_stage1_training_curves.png", dpi=120, bbox_inches="tight")
plt.show(); print("Saved cnn_stage1_training_curves.png")


In [ ]:
# ── Stage-1 Test Set Evaluation ──────────────────────────────────────────
s1_test, latency_map = {}, {}
N_BENCH = min(1000, len(X_test_s))
X_bench = X_test_s[:N_BENCH]

for name, res in s1_val.items():
    if name == "CNN":
        y_pred, y_proba = predict_cnn(cnn_s1, X_test_s, task="binary")
        # latency
        _, _ = predict_cnn(cnn_s1, X_bench, task="binary")
        t0 = time.perf_counter()
        for _ in range(5): predict_cnn(cnn_s1, X_bench, task="binary")
        latency_map["CNN"] = (time.perf_counter()-t0)/(5*N_BENCH)*1e6
    else:
        model = res["model"]
        y_pred  = model.predict(X_test_s)
        y_proba = (model.predict_proba(X_test_s)[:,1]
                   if hasattr(model,"predict_proba")
                   else model.decision_function(X_test_s))
        model.predict(X_bench[:10])
        t0 = time.perf_counter()
        for _ in range(5): model.predict(X_bench)
        latency_map[name] = (time.perf_counter()-t0)/(5*N_BENCH)*1e6

    s1_test[name] = dict(
        f1_attack =f1_score(y_test, y_pred, pos_label=0),
        f1_normal =f1_score(y_test, y_pred, pos_label=1),
        f1_macro  =f1_score(y_test, y_pred, average="macro"),
        auc_roc   =roc_auc_score(y_test, y_proba),
        latency_us=latency_map[name],
        y_pred=y_pred, y_proba=y_proba,
    )
    print(f"\n── {name} (TEST) {'─'*(32-len(name))}")
    print(classification_report(y_test, y_pred,
          target_names=["Attack(0)","Normal(1)"]))


In [ ]:
# ── Comparison plots ─────────────────────────────────────────────────────
names = list(s1_test.keys())
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
x = np.arange(len(names)); w = 0.35
axes[0].bar(x-w/2, [s1_test[n]["f1_attack"] for n in names], w,
            label="Attack(0)", color="#E74C3C", alpha=0.85)
axes[0].bar(x+w/2, [s1_test[n]["f1_normal"] for n in names], w,
            label="Normal(1)", color="#2ECC71", alpha=0.85)
axes[0].set_xticks(x); axes[0].set_xticklabels(names)
axes[0].set_ylim(0.96, 1.005); axes[0].set_ylabel("F1-Score")
axes[0].set_title("Stage 1 — Per-Class F1 (test)", fontweight="bold"); axes[0].legend()
axes[0].yaxis.set_major_formatter(mticker.FormatStrFormatter("%.3f"))

clrs = ["#3498DB","#E67E22","#9B59B6","#2ECC71","#E74C3C"]
for n, cl in zip(names, clrs):
    axes[1].scatter(latency_map[n], s1_test[n]["f1_macro"],
                    s=200, color=cl, zorder=5, edgecolors="white", lw=1.5, label=n)
    axes[1].annotate(n, (latency_map[n], s1_test[n]["f1_macro"]),
                     textcoords="offset points", xytext=(6,4),
                     fontsize=10, fontweight="bold")
axes[1].set_xscale("log"); axes[1].set_xlabel("Latency (μs, log scale)")
axes[1].set_ylabel("F1-Macro"); axes[1].legend(fontsize=9)
axes[1].set_title("Latency vs Accuracy", fontweight="bold")
plt.suptitle("Stage 1 — Binary Classification", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig(OUT/"stage1_comparison.png", dpi=120, bbox_inches="tight")
plt.show(); print("Saved stage1_comparison.png")

# Confusion matrices
fig, axes = plt.subplots(1, 5, figsize=(22, 4))
for ax, name in zip(axes, names):
    cm = confusion_matrix(y_test, s1_test[name]["y_pred"])
    ConfusionMatrixDisplay(cm, display_labels=["Attack","Normal"]).plot(
        ax=ax, colorbar=False, cmap="Blues")
    ax.set_title(name, fontweight="bold")
plt.suptitle("Stage 1 Confusion Matrices", fontsize=12, fontweight="bold")
plt.tight_layout()
plt.savefig(OUT/"stage1_confusion_matrices.png", dpi=120, bbox_inches="tight")
plt.show(); print("Saved stage1_confusion_matrices.png")


In [ ]:
# ════════════ SAVE: stage1_binary_results.csv ════════════
rows = []
for name in s1_test:
    rows.append({"model":name, "stage":"Stage1_Binary",
        "f1_attack_0": round(s1_test[name]["f1_attack"],  4),
        "f1_normal_1": round(s1_test[name]["f1_normal"],  4),
        "f1_macro":    round(s1_test[name]["f1_macro"],   4),
        "auc_roc":     round(s1_test[name]["auc_roc"],    4),
        "latency_us":  round(latency_map[name],           3),
        "train_time_s":round(s1_val[name]["train_time"],  1),
    })
pd.DataFrame(rows).to_csv(OUT/"stage1_binary_results.csv", index=False)
print("✅ Saved stage1_binary_results.csv")

# ════════════ SAVE: stage1_per_class_report.csv ════════════
report_rows = []
for name in s1_test:
    yp = s1_test[name]["y_pred"]
    for lv, ln in [(0,"Attack(0)"), (1,"Normal(1)")]:
        report_rows.append({"model":name, "class":ln,
            "precision": round(precision_score(y_test,yp,pos_label=lv,average="binary"),4),
            "recall":    round(recall_score(y_test,yp,pos_label=lv,average="binary"),4),
            "f1_score":  round(f1_score(y_test,yp,pos_label=lv,average="binary"),4),
            "support":   int((y_test==lv).sum()),
        })
pd.DataFrame(report_rows).to_csv(OUT/"stage1_per_class_report.csv", index=False)
print("✅ Saved stage1_per_class_report.csv")

# ════════════ SAVE: stage1_predictions.csv ════════════
pred_df = X_test.copy().reset_index(drop=True)
pred_df["true_label"] = y_test.values
for name in s1_test:
    pred_df[f"pred_{name}"]  = s1_test[name]["y_pred"]
    pred_df[f"proba_{name}"] = s1_test[name]["y_proba"].round(4)
pred_df.to_csv(OUT/"stage1_predictions.csv", index=False)
print(f"✅ Saved stage1_predictions.csv  ({len(pred_df):,} rows)")


## Section 5 — Stage 2: Attack Type Identification

In [ ]:
if y2_sub is None:
    print("⚠️  Attack_subType not in df2 — skipping Stage 2")
else:
    # ── HGB Stage 2 ──────────────────────────────────────────────────────
    print("Training Stage-2 HGB …")
    t0 = time.time()
    hgb_s2 = HistGradientBoostingClassifier(
        max_iter=200, max_depth=10, learning_rate=0.05, random_state=RANDOM_STATE)
    hgb_s2.fit(X2tr_s, y2tr_enc)
    hgb_s2_time = time.time()-t0
    y2te_hgb = hgb_s2.predict(X2te_s)
    hgb_f1   = f1_score(y2te_enc, y2te_hgb, average="macro")
    print(f"HGB Stage-2 F1-Macro: {hgb_f1:.4f}  ({hgb_s2_time:.1f}s)")
    print(classification_report(y2te_enc, y2te_hgb, target_names=le.classes_))


In [ ]:
if y2_sub is not None:
    # ── CNN Stage 2 (PyTorch) ────────────────────────────────────────────
    cnn_s2     = CNN1D(X2tr_s.shape[1], n_out=N_CLASSES2)
    tr2_loader = make_loader(X2tr_s,  y2tr_enc.astype(np.int64),  batch_size=1024)
    vl2_loader = make_loader(X2val_s, y2val_enc.astype(np.int64), batch_size=1024, shuffle=False)

    print("Training Stage-2 CNN …")
    t0 = time.time()
    hist_s2 = train_cnn(cnn_s2, tr2_loader, vl2_loader,
                        epochs=30, lr=1e-3, patience=5, task="multiclass")
    cnn_s2_time = time.time()-t0
    print(f"\nDone in {cnn_s2_time:.1f}s  ({len(hist_s2['train_loss'])} epochs)")

    y2te_cnn_pred, y2te_cnn_proba = predict_cnn(cnn_s2, X2te_s, task="multiclass")
    cnn_s2_f1 = f1_score(y2te_enc, y2te_cnn_pred, average="macro")
    print(f"CNN Stage-2 F1-Macro: {cnn_s2_f1:.4f}")
    print(classification_report(y2te_enc, y2te_cnn_pred, target_names=le.classes_))


In [ ]:
if y2_sub is not None:
    # Training curves
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    ep2 = range(1, len(hist_s2["train_loss"])+1)
    axes[0].plot(ep2, hist_s2["train_loss"], label="train", color="#E74C3C")
    axes[0].plot(ep2, hist_s2["val_loss"],   label="val",   color="#3498DB")
    axes[0].set_title("CNN Stage 2 — Loss", fontweight="bold"); axes[0].legend()
    axes[1].plot(ep2, hist_s2["val_metric"], color="#2ECC71")
    axes[1].set_title("CNN Stage 2 — Val F1-Macro", fontweight="bold")
    plt.suptitle("Stage 2 CNN Training Curves", fontsize=13, fontweight="bold")
    plt.tight_layout()
    plt.savefig(OUT/"cnn_stage2_training_curves.png", dpi=120, bbox_inches="tight")
    plt.show(); print("Saved cnn_stage2_training_curves.png")

    # Confusion matrix
    cm_s2 = confusion_matrix(y2te_enc, y2te_cnn_pred)
    fig, ax = plt.subplots(figsize=(11, 8))
    sns.heatmap(cm_s2, annot=True, fmt="d", cmap="Blues",
                xticklabels=le.classes_, yticklabels=le.classes_, ax=ax)
    ax.set_title("Stage 2 CNN — Confusion Matrix", fontweight="bold")
    plt.xticks(rotation=45, ha="right"); plt.tight_layout()
    plt.savefig(OUT/"stage2_confusion_matrix.png", dpi=120, bbox_inches="tight")
    plt.show(); print("Saved stage2_confusion_matrix.png")


In [ ]:
if y2_sub is not None:
    # ════════════ SAVE: stage2_subtype_report.csv ════════════
    rpt_cnn = classification_report(y2te_enc, y2te_cnn_pred,
                  target_names=le.classes_, output_dict=True)
    rpt_hgb = classification_report(y2te_enc, y2te_hgb,
                  target_names=le.classes_, output_dict=True)
    sub_rows = []
    for cls, mets in rpt_cnn.items():
        if isinstance(mets, dict):
            hm = rpt_hgb.get(cls, {})
            sub_rows.append({"class":cls,
                "cnn_precision":round(mets["precision"],4),
                "cnn_recall":   round(mets["recall"],4),
                "cnn_f1":       round(mets["f1-score"],4),
                "hgb_precision":round(hm.get("precision",0),4),
                "hgb_recall":   round(hm.get("recall",0),4),
                "hgb_f1":       round(hm.get("f1-score",0),4),
                "support":      mets["support"],
            })
    pd.DataFrame(sub_rows).to_csv(OUT/"stage2_subtype_report.csv", index=False)
    print("✅ Saved stage2_subtype_report.csv")

    # ════════════ SAVE: stage2_predictions.csv ════════════
    s2_df = X2te.copy().reset_index(drop=True)
    s2_df["true_subtype"]     = le.inverse_transform(y2te_enc)
    s2_df["cnn_pred_subtype"] = le.inverse_transform(y2te_cnn_pred)
    s2_df["hgb_pred_subtype"] = le.inverse_transform(y2te_hgb)
    s2_df["cnn_correct"]      = (y2te_cnn_pred == y2te_enc).astype(int)
    s2_df["hgb_correct"]      = (y2te_hgb      == y2te_enc).astype(int)
    s2_df.to_csv(OUT/"stage2_predictions.csv", index=False)
    print(f"✅ Saved stage2_predictions.csv  ({len(s2_df):,} rows)")


## Section 6 — Full Two-Stage Pipeline (End-to-End)

In [ ]:
if y2_sub is not None:
    best_s1 = max(s1_test, key=lambda n: s1_test[n]["f1_macro"])
    print(f"Best Stage-1 model: {best_s1}  "
          f"(F1-Macro={s1_test[best_s1]['f1_macro']:.4f})")

    s1_pred  = s1_test[best_s1]["y_pred"]
    s1_proba = s1_test[best_s1]["y_proba"]
    attack_mask = (s1_pred == 0)
    print(f"Samples flagged ATTACK: {attack_mask.sum():,} "
          f"({attack_mask.mean()*100:.1f}%)")

    # Stage 2 on attack-flagged samples only
    X_atk = X_test_s[attack_mask]
    if len(X_atk) > 0:
        s2_pred_atk, _ = predict_cnn(cnn_s2, X_atk, task="multiclass")
        s2_labels_atk  = le.inverse_transform(s2_pred_atk)
    else:
        s2_labels_atk = np.array([])

    # Assemble full result
    pipe_df = X_test.copy().reset_index(drop=True)
    pipe_df["true_binary_label"] = y_test.values
    pipe_df["stage1_pred"]       = s1_pred
    pipe_df["stage1_proba"]      = s1_proba.round(4)
    pipe_df["stage1_decision"]   = np.where(s1_pred==0, "ATTACK", "NORMAL")
    pipe_df["stage2_attack_type"]= "N/A"

    atk_idx = np.where(attack_mask)[0]
    for i, lbl in zip(atk_idx, s2_labels_atk):
        pipe_df.at[i, "stage2_attack_type"] = lbl

    pipe_df["final_prediction"] = pipe_df.apply(
        lambda r: r["stage2_attack_type"] if r["stage1_decision"]=="ATTACK" else "NORMAL",
        axis=1)

    # ════════════ SAVE: pipeline_combined_predictions.csv ════════════
    pipe_df.to_csv(OUT/"pipeline_combined_predictions.csv", index=False)
    print(f"\n✅ Saved pipeline_combined_predictions.csv  ({len(pipe_df):,} rows)")
    print("\nFinal prediction distribution:")
    print(pipe_df["final_prediction"].value_counts().to_string())


## Section 7 — Feature Importance

In [ ]:
rf_model = s1_val["RF"]["model"]
imp_rf   = pd.Series(rf_model.feature_importances_,
                     index=USE_FEATS).sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(9, 7))
imp_rf.sort_values().plot(kind="barh", ax=ax,
    color=["#E74C3C" if v>imp_rf.median() else "#85C1E9" for v in imp_rf.sort_values()],
    edgecolor="white")
ax.set_title("Random Forest Feature Importances", fontsize=12, fontweight="bold")
plt.tight_layout()
plt.savefig(OUT/"feature_importance_rf.png", dpi=120, bbox_inches="tight")
plt.show()

fi_df = imp_rf.reset_index()
fi_df.columns = ["feature","importance"]
fi_df["rank"] = range(1, len(fi_df)+1)
fi_df.to_csv(OUT/"feature_importance_rf.csv", index=False)
print("✅ Saved feature_importance_rf.csv")
print(fi_df.head(10).to_string(index=False))


## Section 8 — Per-Device Evaluation

In [ ]:
if "Device_Name" in df2.columns:
    use2c = [f for f in USE_FEATS if f in df2.columns]
    best_sk = max((n for n in s1_val if n!="CNN"),
                  key=lambda n: s1_val[n]["f1_macro"])
    device_rows = []
    for device in sorted(df2["Device_Name"].unique()):
        mask  = df2["Device_Name"] == device
        X_dev = scaler1.transform(log1p_tf(df2.loc[mask, use2c], SKEW_FEATS))
        y_dev = df2.loc[mask, "label"]
        if len(y_dev.unique()) < 2: continue
        ydev_pred = s1_val[best_sk]["model"].predict(X_dev)
        device_rows.append({"device":device, "model":best_sk,
            "f1_attack": round(f1_score(y_dev,ydev_pred,pos_label=0,zero_division=0),4),
            "f1_normal": round(f1_score(y_dev,ydev_pred,pos_label=1,zero_division=0),4),
            "n_samples": int(mask.sum()),
        })
    df_dev = pd.DataFrame(device_rows).sort_values("f1_attack", ascending=False)
    df_dev.to_csv(OUT/"per_device_results.csv", index=False)
    print("✅ Saved per_device_results.csv")
    print(df_dev.to_string(index=False))

    fig, ax = plt.subplots(figsize=(12, 5))
    xd = np.arange(len(df_dev))
    ax.bar(xd-0.2, df_dev["f1_attack"], 0.4, label="F1-Attack", color="#E74C3C", alpha=0.85)
    ax.bar(xd+0.2, df_dev["f1_normal"], 0.4, label="F1-Normal", color="#2ECC71", alpha=0.85)
    ax.set_xticks(xd)
    ax.set_xticklabels(df_dev["device"], rotation=30, ha="right", fontsize=8)
    ax.set_ylim(0, 1.05); ax.set_ylabel("F1-Score")
    ax.set_title("Per-Device F1", fontweight="bold"); ax.legend()
    plt.tight_layout()
    plt.savefig(OUT/"per_device_results.png", dpi=120, bbox_inches="tight")
    plt.show(); print("Saved per_device_results.png")
else:
    print("⚠️  Device_Name not in df2 — skipping")


## Section 9 — Save CNN History & Final Summary

In [ ]:
# ════════════ SAVE: cnn_training_history.csv ════════════
hist_rows = []
for ep_i, (tl, vl, vm) in enumerate(zip(
        hist_s1["train_loss"], hist_s1["val_loss"], hist_s1["val_metric"])):
    hist_rows.append({"stage":"Stage1_Binary","epoch":ep_i+1,
        "train_loss":round(tl,5),"val_loss":round(vl,5),"val_f1_macro":round(vm,5)})

if y2_sub is not None:
    for ep_i, (tl, vl, vm) in enumerate(zip(
            hist_s2["train_loss"], hist_s2["val_loss"], hist_s2["val_metric"])):
        hist_rows.append({"stage":"Stage2_Multiclass","epoch":ep_i+1,
            "train_loss":round(tl,5),"val_loss":round(vl,5),"val_f1_macro":round(vm,5)})

pd.DataFrame(hist_rows).to_csv(OUT/"cnn_training_history.csv", index=False)
print("✅ Saved cnn_training_history.csv")


In [ ]:
# ════════════ FINAL SUMMARY ════════════
print("="*65)
print("FINAL RESULTS — TEST SET")
print("="*65)
final = pd.DataFrame({
    name: {"F1-Attack(0)": round(s1_test[name]["f1_attack"],4),
           "F1-Normal(1)": round(s1_test[name]["f1_normal"],4),
           "F1-Macro":     round(s1_test[name]["f1_macro"],4),
           "AUC-ROC":      round(s1_test[name]["auc_roc"],4),
           "Latency(μs)":  round(latency_map[name],3)}
    for name in s1_test
}).T
print(final.to_string())

if y2_sub is not None:
    print(f"\nStage-2 CNN F1-Macro : {cnn_s2_f1:.4f}")
    print(f"Stage-2 HGB F1-Macro : {hgb_f1:.4f}")

print("\n── Output files saved ────────────────────────────────────")
for f in sorted(OUT.glob("*")):
    print(f"  {f.name:50s}  {f.stat().st_size/1e3:6.1f} KB")
